In [54]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


import warnings

warnings.simplefilter("ignore", FutureWarning)

pd.set_option('display.max_columns', None)

In [55]:
cols = [
    'year', 'month', 'day', 'hour', 'minute', 'second', # A–F
    'glucose_level', # G
    'finger_stick', # H
    'basal', # I
    'bolus', # J
    'sleep', # K
    'work', # L
    'stressors', # M
    'hypo_event', # N
    'illness', # O
    'exercise', # P
    'basis_heart_rate', # Q
    'basis_gsr', # R
    'basis_skin_temperature', # S
    'basis_air_temperature',  # T
    'basis_step', # U
    'basis_sleep', # V
    'meal', # W
    'meal_type' # X
]

In [56]:
# Definicio de id dels pacients
PACIENTS = [559, 563, 570, 575, 588, 591]

# Definició de l'horitzo
HORITZO = {30:6, 60:12}  # minuts : pas/files (de 5 mins)

In [57]:
# Funcio per obtenir els datasets
def load_data(id: int, train_or_test: str) -> pd.DataFrame:
    df=pd.read_csv(f'../data/{id}/{id}_{train_or_test}.csv', sep=';', header = None, names = cols)
    return df

prova_559_train = load_data(559, 'train')
prova_559_test = load_data(559, 'test')

In [58]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    prep = df.copy()

    prep['timestamp'] = pd.to_datetime(
        dict(year=df.year, month=df.month, day=df.day, hour=df.hour, minute=df.minute)
    )
    
    prep.sort_values('timestamp', inplace=True)

    # Coma decimal a punt
    convert = ["basal","bolus","basis_gsr","basis_skin_temperature","basis_air_temperature"]
    
    for c in convert:
        prep[c] = (prep[c].astype(str)
                   .str.replace(",",".", regex=False)
                   .str.strip()
                   .astype(float))

    # Unifiquem tipo
    cat_meal = {
        1:"Desayuno",
        2:"Almuerzo",
        3:"Cena",
        4:"Snack",
        5:"Correccion_hipo"
    }

    prep["meal_type"] = prep["meal_type"].map(cat_meal).astype("category")
    prep = pd.get_dummies(prep, columns=['meal_type'], dummy_na=False, prefix='meal')

    # Drop columnas amb casi tot NaN o valor constant
    prep = prep.drop(columns=["second","finger_stick","meal"])

    # Zeros que no poden ser valids
    invalid_zero = [
        "glucose_level",
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_air_temperature"
    ]
    
    prep[invalid_zero] = prep[invalid_zero].replace(0, np.nan)
    prep[invalid_zero] = prep[invalid_zero].fillna(method='ffill')

    prep = prep.dropna(subset=['glucose_level'])

    prep = prep.drop(columns=['timestamp'])
    return prep

In [59]:
prova_559_train = preprocess(prova_559_train)
prova_559_test = preprocess(prova_559_test)

print('Shape train: ', prova_559_train.shape)
print('Shape test: ', prova_559_test.shape)
print('\n')
print(prova_559_train.isnull().mean()*100)

Shape train:  (12081, 25)
Shape test:  (2876, 24)


year                      0.000000
month                     0.000000
day                       0.000000
hour                      0.000000
minute                    0.000000
glucose_level             0.000000
basal                     0.000000
bolus                     0.000000
sleep                     0.000000
work                      0.000000
stressors                 0.000000
hypo_event                0.000000
illness                   0.000000
exercise                  0.000000
basis_heart_rate          1.158844
basis_gsr                 1.158844
basis_skin_temperature    1.158844
basis_air_temperature     1.158844
basis_step                0.000000
basis_sleep               0.000000
meal_Almuerzo             0.000000
meal_Cena                 0.000000
meal_Correccion_hipo      0.000000
meal_Desayuno             0.000000
meal_Snack                0.000000
dtype: float64


In [60]:
def make_xy(df: pd.DataFrame, steps: int):

    y = df['glucose_level'].shift(-steps)

    X = df.iloc[:-steps].copy() # totes les columnes, inclosa 'glucose_level'
    y = y.iloc[:-steps] # mateixes files que X

    return X, y



In [61]:
def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

In [62]:
# ------------------------------------------------
# Entrenamiento, predicción y métrica paciente por paciente
# ------------------------------------------------
all_metrics = []

for id in PACIENTS:
    print(f'\nPaciente {id}')
    train_raw = load_data(id, 'train')
    test_raw  = load_data(id, 'test')

    train = preprocess(train_raw)
    test  = preprocess(test_raw)

    
    # igualamos columnas de train y test (outer join) y rellenamos lo que falte con 0
    train, test = train.align(test, join='outer', axis=1, fill_value=0)

    # ignorar los primeros 60 min del test
    test = test.iloc[12:].reset_index(drop=True)

    for minuts, steps in HORITZO.items():
        # ---------- entrenamiento offline ----------
        X_train, y_train = make_xy(train, steps)

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)

        # ---------- predicción en test ----------
        X_test = test.iloc[:-steps].copy()

        y_true = test['glucose_level'].iloc[:-steps].reset_index(drop=True)
        y_pred = model.predict(X_test)

        # guardar CSV requerido
        out_csv = f'../data/predicted/pred{id}_{minuts}min.csv'

        df_pred = pd.DataFrame({'idx_original': X_test.index, f'pred_glucose_t+{minuts}': y_pred})
        df_pred.to_csv(out_csv, index=False)

        # ---------- métricas ----------
        rmse, mae = evaluate(y_true, y_pred)
        all_metrics.append({
            'Paciente': id,
            'Horizonte': f'{minuts} min',
            'RMSE': rmse,
            'MAE' : mae
        })
        print(f'{minuts} min: RMSE={rmse:.2f}  MAE={mae:.2f}')




Paciente 559
30 min: RMSE=16.28  MAE=11.53
60 min: RMSE=29.02  MAE=22.23

Paciente 563
30 min: RMSE=10.70  MAE=7.40
60 min: RMSE=25.21  MAE=17.15

Paciente 570
30 min: RMSE=11.92  MAE=8.52
60 min: RMSE=22.81  MAE=17.57

Paciente 575
30 min: RMSE=43.34  MAE=22.23
60 min: RMSE=42.21  MAE=31.48

Paciente 588
30 min: RMSE=14.02  MAE=9.85
60 min: RMSE=25.11  MAE=18.99

Paciente 591
30 min: RMSE=18.95  MAE=12.99
60 min: RMSE=36.04  MAE=28.36


In [63]:
# ------------------------------------------------
# Tabla final (promedio incluido)
# ------------------------------------------------
metrics_df = pd.DataFrame(all_metrics)
prom = metrics_df.groupby('Horizonte')[['RMSE','MAE']].mean().reset_index()
prom.insert(0, 'Paciente', 'PROMEDIO')
result = pd.concat([metrics_df, prom], ignore_index=True)
print('\n==== RESULTADOS FINALES ====')
print(result)


==== RESULTADOS FINALES ====
    Paciente Horizonte       RMSE        MAE
0        559    30 min  16.275801  11.534473
1        559    60 min  29.021431  22.230445
2        563    30 min  10.704493   7.400236
3        563    60 min  25.210563  17.147045
4        570    30 min  11.923503   8.522431
5        570    60 min  22.811173  17.570797
6        575    30 min  43.344409  22.232761
7        575    60 min  42.205782  31.479384
8        588    30 min  14.018158   9.851504
9        588    60 min  25.107083  18.988331
10       591    30 min  18.945540  12.987004
11       591    60 min  36.044227  28.358751
12  PROMEDIO    30 min  19.201984  12.088068
13  PROMEDIO    60 min  30.066710  22.629126
